
### Notebook : 01_shared_models




##### 1 : Purpose

This notebook defines the reusable shared types (Literal definitions) and shared Pydantic models (schemas) that will be used throughout the multi-agent workflow.


It contains:

- Shared type definitions
- Coordinator task schema
- Base agent-result schema
- Specialist-agent result schemas
- Execution-history schema

It does not contain:

- Shared-state creation
- Helper functions
- Tool calls
- Agent execution logic
- Tests that automatically run


##### 2 : Technologies Used

- Python
- Pydantic
- Type hints
- Literal types
- Inheritance
- UTC timestamps
- Databricks notebooks


##### 3: Input

- No customer request or tool output is processed directly.

- The notebook receives only the schema requirements of the multi-agent system.


##### 4 : Output

- Reusable Python types and Pydantic models available to all downstream notebooks.


##### 5 : Multi-Agent Architecture

The multi-agent system separates planning, specialized execution, retention
decision-making, and final response generation.

```text

Coordinator and specialist agents
              │
              ▼
      Shared Pydantic schemas
              │
      ┌───────┼───────────┐
      ▼       ▼           ▼
Execution   Agent       Execution
plans       results     records
              │
              ▼
Validated data exchanged across notebooks

```


###### 6 : Imports

In [0]:
from datetime import datetime, timezone
from typing import Any, Dict, List, Literal, Optional

from pydantic import BaseModel, Field


###### 7 : Shared type definitions

In [0]:
# ===============
# Workflow Types
# ===============

AgentName = Literal[
    "coordinator_agent",
    "sql_agent",
    "prediction_agent",
    "vector_search_agent",
    "retention_agent",
    "final_response_agent",
]

AgentStatus = Literal[
    "pending",
    "running",
    "success",
    "failed",
    "skipped",
]

RequestType = Literal[
    "sql_analytics",
    "prediction",
    "vector_search",
    "retention",
    "combined",
    "unsupported",
]

# ===============
# Business Types
# ===============

SupportCategory = Literal[
    "Billing",
    "Technical",
    "Service",
    "Account",
    "General",
]

RetentionAction = Literal[
    "offer_discount",
    "offer_support_package",
    "service_quality_review",
    "billing_review",
    "no_action",
]


##### 8 : Shared Pydantic Schemas

In [0]:
# 1 : Coordinator task schema

class AgentTask(BaseModel):
    """
    Represents one task in the Coordinator Agent's execution plan.
    """

    task_id: str

    agent_name: AgentName

    task_description: str

    depends_on: List[AgentName] = Field(
        default_factory=list
    )

In [0]:
# Example

example_task = AgentTask(
    task_id="task_1",
    agent_name="prediction_agent",
    task_description=(
        "Predict the support-ticket category for the customer issue."
    ),
    depends_on=[],
)

print(example_task.model_dump())

In [0]:
# Expected Stucture

{
    "task_id": "task_1",
    "agent_name": "prediction_agent",
    "task_description": (
        "Predict the support-ticket category for the customer issue."
    ),
    "depends_on": [],
}

In [0]:
# 2 : Base agent-result schema

class BaseAgentResult(BaseModel):
    """
    Common fields returned by every agent.
    """

    agent_name: AgentName

    status: AgentStatus

    message: str

    task_description: Optional[str] = None

    error: Optional[str] = None

In [0]:
# 3 : Coordinator result schema

class CoordinatorResult(BaseAgentResult):
    """
    Validated result returned by the Coordinator Agent.
    """

    agent_name: Literal["coordinator_agent"] = (
        "coordinator_agent"
    )

    request_type: RequestType

    reasoning: str

    tasks: List[AgentTask] = Field(
        default_factory=list
    )

In [0]:
# Example

example_coordinator_result = CoordinatorResult(
    status="success",
    message="Execution plan created successfully.",
    task_description=(
        "Analyze the customer request and create an execution plan."
    ),
    request_type="prediction",
    reasoning=(
        "The request asks the system to predict a support-ticket "
        "category."
    ),
    tasks=[
        AgentTask(
            task_id="task_1",
            agent_name="prediction_agent",
            task_description=(
                "Predict the category of the customer issue."
            ),
        ),
        AgentTask(
            task_id="task_2",
            agent_name="final_response_agent",
            task_description=(
                "Generate the final response using the prediction result."
            ),
            depends_on=["prediction_agent"],
        ),
    ],
)

print(example_coordinator_result.model_dump())

In [0]:
# 4 : SQL Agent result schema

class SQLAgentResult(BaseAgentResult):
    """
    Validated result returned by the SQL Agent.
    """

    agent_name: Literal["sql_agent"] = "sql_agent"

    sql_action: Optional[str] = None

    sql_result: Any = None

    row_count: Optional[int] = None

# The sql_result field uses Any because the SQL tool may return: Spark Row objects, Lists of rows, Dictionaries, Numeric aggregates, Tabular results

In [0]:
# 5 : Prediction Agent result schema

class PredictionAgentResult(BaseAgentResult):
    """
    Validated result returned by the Prediction Agent.
    """

    agent_name: Literal["prediction_agent"] = (
        "prediction_agent"
    )

    predicted_category: Optional[SupportCategory] = None

    confidence: Optional[float] = None

    model_name: Optional[str] = None

    raw_prediction: Any = None

# The raw_prediction field preserves the original Prediction Tool output when needed.

In [0]:
#Example
example_prediction_result = PredictionAgentResult(
    status="success",
    message="Prediction completed successfully.",
    task_description=(
        "Predict the category of the customer issue."
    ),
    predicted_category="Technical",
    confidence=0.91,
    model_name="support-ticket-classifier",
    raw_prediction={
        "prediction": "Technical",
        "confidence": 0.91,
    },
)

print(example_prediction_result.model_dump())

In [0]:
# 6 : Vector Search Agent result schema

class VectorSearchAgentResult(BaseAgentResult):
    """
    Validated result returned by the Vector Search Agent.
    """

    agent_name: Literal["vector_search_agent"] = (
        "vector_search_agent"
    )

    query_text: Optional[str] = None

    similar_tickets: List[Dict[str, Any]] = Field(
        default_factory=list
    )

    result_count: int = 0

In [0]:
# Example

{
    "ticket_id": "T010",
    "issue": "Internet connection is very slow.",
    "category": "Technical",
    "similarity_score": 0.94,
}

In [0]:
# 7 : Retention Agent result schema

class RetentionAgentResult(BaseAgentResult):
    """
    Validated result returned by the Retention Agent.
    """

    agent_name: Literal["retention_agent"] = (
        "retention_agent"
    )

    recommended_action: Optional[RetentionAction] = None

    recommendation: Optional[str] = None

    prediction_context: Optional[Dict[str, Any]] = None

    similar_ticket_context: List[Dict[str, Any]] = Field(
        default_factory=list
    )

In [0]:
# 8 : Final Response Agent result schema

class FinalResponseAgentResult(BaseAgentResult):
    """
    Validated result returned by the Final Response Agent.
    """

    agent_name: Literal["final_response_agent"] = (
        "final_response_agent"
    )

    final_response: Optional[str] = None

In [0]:
class AgentErrorRecord(BaseModel):
    """
    Represents one agent or workflow error.
    """

    agent_name: AgentName

    error_code: str

    error_message: str

    timestamp: datetime = Field(
        default_factory=lambda: datetime.now(
            timezone.utc
        )
    )


##### 9 : ExecutionRecord

In [0]:
# Execution-history schema: An ExecutionRecord represents one event in the workflow execution history.

class ExecutionRecord(BaseModel):
    """
    Represents one agent execution event in the multi-agent workflow.
    """

    agent_name: AgentName

    status: AgentStatus

    message: str

    timestamp: datetime = Field(
        default_factory=lambda: datetime.now(timezone.utc)
    )

In [0]:
# Example

example_execution_record = ExecutionRecord(
    agent_name="prediction_agent",
    status="success",
    message="Prediction task completed successfully.",
)

print(example_execution_record.model_dump())

In [0]:
# Expected Structure

{
    "agent_name": "prediction_agent",
    "status": "success",
    "message": "Prediction task completed successfully.",
    "timestamp": datetime(...),
}


##### 10 : Schema Validation

In [0]:
def validate_shared_schemas() -> None:
    """
    Run basic validation checks for shared schemas.
    Call manually only when testing this notebook.
    """

    example_task = AgentTask(
        task_id="task_1",
        agent_name="prediction_agent",
        task_description=(
            "Predict the support-ticket category."
        ),
    )

    example_prediction_result = PredictionAgentResult(
        status="success",
        message="Prediction completed successfully.",
        predicted_category="Technical",
        confidence=0.91,
    )

    example_execution_record = ExecutionRecord(
        agent_name="prediction_agent",
        status="success",
        message="Prediction task completed successfully.",
    )

    assert example_task.agent_name == "prediction_agent"
    assert (
        example_prediction_result.predicted_category
        == "Technical"
    )
    assert example_execution_record.status == "success"

    print("All shared-schema validation checks passed.")

In [0]:
schema_names = [
    AgentTask.__name__,
    BaseAgentResult.__name__,
    CoordinatorResult.__name__,
    SQLAgentResult.__name__,
    PredictionAgentResult.__name__,
    VectorSearchAgentResult.__name__,
    RetentionAgentResult.__name__,
    FinalResponseAgentResult.__name__,
    ExecutionRecord.__name__,
    AgentErrorRecord.__name__,
]

print("Schemas loaded successfully:")

for schema_name in schema_names:
    print(f"- {schema_name}")


##### 11 : Key learnings

• A multi-agent system divides a complex workflow among specialized agents.

• Each agent should have one clearly defined responsibility.

• Shared Literal types restrict workflow and business values to approved options.

• Pydantic models create explicit communication contracts between agents.

• Schema inheritance avoids repeating common agent-result fields.

• Agent dependencies describe the required execution order.

• Pydantic validates agent outputs before downstream components use them.

• Structured agent results should remain separate from the final customer-facing response.

• Consistent agent names, statuses, messages, and errors make workflows  easier to test and debug.

• Timezone-aware UTC timestamps provide consistent execution records.



##### 12 : Conclusion

This notebook established the shared data contracts for the multi-agent customer-support system.

The architecture now contains:

- Standardized agent names, statuses, and request types
- Valid support categories and retention actions
- Dependency-aware Coordinator tasks
- A reusable base result model
- Validated Coordinator and specialist-agent result schemas
- A validated execution-history record schema

These shared types and Pydantic models ensure that all downstream notebooks exchange information using consistent and validated structures. Shared-state creation and reusable state-management functions will be implemented in the next notebook.


##### 13 : Next Notebook

Next notebook defines the shared workflow state and reusable state-management functions.